In [3]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [4]:
# ==========================================
# 1. LOAD DATA (already preprocessed, deduplicated, and split)
# ==========================================
TRAIN_FILE = "TrainingDataset.csv"
VAL_FILE = "ValidationDataset.csv"
TEST_FILE = "TestingDataset.csv"
TEXT_COL = 0
LABEL_COL = 1

train_df = pd.read_csv( TRAIN_FILE, header=None)
val_df = pd.read_csv( VAL_FILE, header=None)
test_df = pd.read_csv( TEST_FILE, header=None)

X_train, y_train = train_df[TEXT_COL], train_df[LABEL_COL]
X_val, y_val = val_df[TEXT_COL], val_df[LABEL_COL]
X_test, y_test = test_df[TEXT_COL], test_df[LABEL_COL]

print("Dataset Sizes Loaded:")
print(f" - Train Set:      {len(X_train)} samples")
print(f" - Validation Set: {len(X_val)} samples")
print(f" - Test Set:       {len(X_test)} samples\n")


Dataset Sizes Loaded:
 - Train Set:      108786 samples
 - Validation Set: 14510 samples
 - Test Set:       21757 samples



In [5]:
# ==========================================
# 2. FEATURE EXTRACTION (TF-IDF) + 3. MODEL (LinearSVC)
# ==========================================
# Plug in an Arabic stopword list here if you have one, e.g.:
# arabic_stopwords = [...]
# TfidfVectorizer(..., stop_words=arabic_stopwords)
arabic_stopwords = None  # replace with your stopword list if available

pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                sublinear_tf=True,          # 1 + log(TF) instead of raw TF
                min_df=2,                   # drop terms in only 1 document (noise/typos)
                max_df=0.85,                # drop terms in >85% of documents (uninformative)
                stop_words=arabic_stopwords,
            ),
        ),
        ("svm", LinearSVC(random_state=42, class_weight="balanced")),
    ]
)

In [6]:
# ==========================================
# 4. HYPERPARAMETER TUNING (5-fold CV on training set only)
# ==========================================
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "svm__C": [0.01, 0.1, 1.0, 10.0],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)

print("Training & Tuning Model on Training Set...")
grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters Found:")
print(grid_search.best_params_)
print(
    f"Best 5-Fold Cross-Validation Macro-F1 (Train Set): {grid_search.best_score_:.4f}\n"
)

best_model = grid_search.best_estimator_

Training & Tuning Model on Training Set...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Hyperparameters Found:
{'svm__C': 1.0, 'tfidf__ngram_range': (1, 2)}
Best 5-Fold Cross-Validation Macro-F1 (Train Set): 0.9049



In [7]:
# ==========================================
# 5. VALIDATION SET EVALUATION
# ==========================================
y_val_pred = best_model.predict(X_val)
val_f1_macro = f1_score(y_val, y_val_pred, average="macro")
val_f1_weighted = f1_score(y_val, y_val_pred, average="weighted")

print("==========================================")
print("VALIDATION SET RESULTS")
print("==========================================")
print(f"Validation Macro-F1:    {val_f1_macro:.4f}")
print(f"Validation Weighted-F1: {val_f1_weighted:.4f}\n")

VALIDATION SET RESULTS
Validation Macro-F1:    0.9068
Validation Weighted-F1: 0.9077



In [8]:
# ==========================================
# 6. FINAL TEST SET EVALUATION (once, on held-out test set)
# ==========================================
y_test_pred = best_model.predict(X_test)
test_f1_macro = f1_score(y_test, y_test_pred, average="macro")
test_f1_weighted = f1_score(y_test, y_test_pred, average="weighted")

print("==========================================")
print("FINAL UNSEEN TEST SET RESULTS")
print("==========================================")
print(f"Test Macro-F1:    {test_f1_macro:.4f}")
print(f"Test Weighted-F1: {test_f1_weighted:.4f}\n")

print("Full Classification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix (rows=true, cols=predicted):")
labels_sorted = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_test_pred, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print(cm_df, "\n")

FINAL UNSEEN TEST SET RESULTS
Test Macro-F1:    0.9102
Test Weighted-F1: 0.9112

Full Classification Report:
                 precision    recall  f1-score   support

         اقتصاد       0.91      0.92      0.91      2239
      تكنولوجيا       0.92      0.91      0.92      2328
خواطر واقتباسات       0.87      0.90      0.88      2447
            دين       0.90      0.90      0.90      1957
          رياضه       0.99      0.98      0.98      2327
          سيارت       0.97      0.95      0.96      2214
          سياسه       0.93      0.94      0.94      2437
            صحه       0.83      0.87      0.85      2099
            طهي       0.96      0.94      0.95      1719
           علوم       0.82      0.79      0.81      1990

       accuracy                           0.91     21757
      macro avg       0.91      0.91      0.91     21757
   weighted avg       0.91      0.91      0.91     21757

Confusion Matrix (rows=true, cols=predicted):
                 اقتصاد  تكنولوجيا  خواطر وا

In [9]:
# ==========================================
# 7. MODEL SAVING
# ==========================================
joblib.dump(best_model, "tfidf_svm_baseline.joblib")
print("Saved fitted pipeline to tfidf_svm_baseline.joblib")

Saved fitted pipeline to tfidf_svm_baseline.joblib


In [11]:
predicted_class_names = y_test_pred

# Create a DataFrame to display test tweets with their predicted classes
results_df = pd.DataFrame({
    'Tweet': X_test.values,
    'Actual_Class': y_test.values,
    'Predicted_Class': predicted_class_names
})

print("Test Tweets with Predicted Classes (First 10 samples):")
results_df

Test Tweets with Predicted Classes (First 10 samples):


,Tweet,Actual_Class,Predicted_Class
0,تشكيله ضمك امام الاتحاد,رياضه,رياضه
1,سبب هذا الاكتشاف الضخم زعزعه وسط العلماء وهذا ...,تكنولوجيا,علوم
2,حذر المركز الأوروبي للوقايه من الأمراض ومكافحت...,صحه,صحه
3,خسوف القلب متي رأيت القلب ترحل عنه حب الله وال...,دين,خواطر واقتباسات
4,تشكيله الفريقين السنغال مصر,رياضه,رياضه
...,...,...,...
21752,هل شعرت ببعض المشاكل في عينك ربما يكون السبب أ...,علوم,علوم
21753,أجمل كيا برأيك,سيارت,سيارت
21754,ضع في اعتبارك دائما أن كل البشر ينتمون إلي أصل...,خواطر واقتباسات,خواطر واقتباسات
21755,نصائح لتسهيل القياده أثناء الأحوال الجويه السيئه,سيارت,سيارت


In [12]:
results_df.to_csv('predicted_tweets.csv', index=False)
print("DataFrame exported to 'predicted_tweets.csv'")

DataFrame exported to 'predicted_tweets.csv'
